# jaxfne Suite No. 4: Oscillatory Push-Pull in a Laminar Network

**Scope:** simulated/proxy readouts for a computational scaffold.

This notebook demonstrates two-level matrix parameter optimization:
- **Outer loop:** AGSDR stochastic candidate proposal
- **Inner loop:** Optax Adam gradient refinement on soft-rate surrogate
- **Final scoring:** declared objective (group firing rates)

Parameters: gAMPA_w and gGABA_w weight matrices, tuned with AGSDR + Optax Adam.

In [ ]:
import jaxfne as jtfne
import optax
import jax.numpy as jnp
import json


## 1. Build the Model

A small E/I network with 20 neurons (16 E, 4 I).

In [ ]:
cfg = (
    jtfne.configuration()
    .network(name="V1", kind="cortical_column", n=20, cell_types={"E": 0.8, "PV": 0.2})
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(domain="laminar_column", conductivity="proxy",
           boundary="mean_zero_neumann", gauge="mean_zero")
    .probe(name="probe", modes=["spikes", "V_m", "CSD"])
)
model = jtfne.construct(cfg)
print("Model constructed:", model.cfg.metadata["truth_mode"])

## 2. Define Objectives and Simulation

Group-rate targets: E group at 12 Hz, I group at 8 Hz.

In [ ]:
n_E = 16  # 80% of 20
n_I = 4   # 20% of 20
objective = jtfne.rate_targets(
    groups={"E": list(range(n_E)), "I": list(range(n_E, n_E + n_I))},
    targets_hz={"E": 12.0, "I": 8.0},
)
sim = jtfne.simulation(duration_ms=1000.0, dt_ms=0.1, seed=0)
print("Simulation: 1000 ms, groups E and I")

## 3. Define Matrix Parameter Specifications

AMPA synaptic weights (excitatory to all neurons) and GABA weights (E→I).

In [ ]:
gAMPA_w_spec = jtfne.matrix_parameter(
    mask="excitatory_to_all",
    bounds=(0.3, 3.0),
)
gGABA_w_spec = jtfne.matrix_parameter(
    mask="E_to_I",
    bounds=(0.1, 5.0),
)
print("gAMPA_w bounds (0.3, 3.0), gGABA_w bounds (0.1, 5.0)")

## 4. Two-Level AGSDR + Adam Optimizer

OUTER: AGSDR proposes multiplicative scale candidates.  
INNER:  refines each candidate using a differentiable
soft-rate sigmoid surrogate loss ( gradient steps).

In [ ]:
optimizer = jtfne.agsdr(
    parameters={
        "gAMPA_w": gAMPA_w_spec,
        "gGABA_w": gGABA_w_spec,
    },
    generations=4,
    population_size=4,
    inner_optimizer=optax.adam(learning_rate=1e-2),
    inner_steps=5,
    seed=42,
)
print("Optimizer:", optimizer.to_dict()["optimizer_class"])
print("Parameters:", list(optimizer.parameters.keys()))
print("Inner optimizer metadata:", json.dumps(optimizer.to_dict()["inner_optimizer"], indent=2))

## 5. Run Matrix Optimization

The two-level loop: AGSDR outer + Adam inner surrogate + real-objective scoring.

In [ ]:
result = model.tune(
    objectives=objective,
    optimizer=optimizer,
    simulation=sim,
)
print("Best score:", result.best_score)
print("Best parameters:", result.best_parameters)
print("Tuning path:", result.summary.get("tuning_path"))

## 6. Inspect and Validate Result

The  contains the tuned weight matrix.  
All output is JSON-safe; no Optax objects are serialized.

In [ ]:
# Verify JSON safety
json_str = json.dumps(result.summary, allow_nan=False)
assert isinstance(json_str, str), "Summary must be JSON-safe"

# Verify both matrix parameters present and 2D
assert "gAMPA_w" in result.best_parameters
assert "gGABA_w" in result.best_parameters
gAMPA = jnp.asarray(result.best_parameters["gAMPA_w"])
gGABA = jnp.asarray(result.best_parameters["gGABA_w"])
assert gAMPA.ndim == 2 and gGABA.ndim == 2
print(f"✓ gAMPA_w {gAMPA.shape}, gGABA_w {gGABA.shape}")

## 7. Summary

| Item | Value |
|------|-------|
| Outer loop | AGSDR stochastic proposal |
| Inner loop | Optax Adam + sigmoid surrogate |
| Final scoring | Real objective (rate targets) |
| Parameters | gAMPA_w (24×24), gGABA_w (24×24) |

**Key design:**
- AGSDR explores candidate scales stochastically
- Adam refines each candidate on differentiable surrogate
- Final score uses real objective, no surrogate
- Result: tuned weight matrix in model + best_parameters